# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}\n")

## 2. Data Overview
Review available record sets and which fields/columns are included. All are referenced by their `@id` for clarity.

Printing key information for each `RecordSet` and its fields.

In [ ]:
# Record sets overview, referenced by `@id`

record_sets = []
if hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet if isinstance(metadata.recordSet, list) else [metadata.recordSet]

print(f"Number of record sets in dataset: {len(record_sets)}\n")
for rs in record_sets:
    rec = dataset.record_set(rs['@id']) if isinstance(rs, dict) and '@id' in rs else dataset.record_set(rs)
    print(f"RecordSet @id: {rec['@id']}")
    if 'name' in rec:
        print(f"  name: {rec['name']}")
    if 'description' in rec:
        print(f"  description: {rec['description']}")
    # List field/column @ids
    print("  Fields (by @id):")
    if 'field' in rec:
        fields = rec['field'] if isinstance(rec['field'], list) else [rec['field']]
        for fld in fields:
            fmeta = None
            try:  # Try getting richer field info
                fmeta = dataset.field(fld['@id']) if isinstance(fld, dict) and '@id' in fld else dataset.field(fld)
            except Exception:
                pass
            print(f"    - {(fld['@id'] if isinstance(fld, dict) and '@id' in fld else fld)} {(f'({fmeta["name"]})' if fmeta and 'name' in fmeta else '')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

> For demonstration, we pick the first available record set.

In [ ]:
# Extract data from each record set
if len(record_sets) == 0:
    print("No record sets found in the dataset.")
else:
    # List of RecordSet @ids
    record_set_ids = []
    for rs in record_sets:
        if isinstance(rs, dict) and '@id' in rs:
            record_set_ids.append(rs['@id'])
        elif isinstance(rs, str):
            record_set_ids.append(rs)
    dataframes = {}
    
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        # Each record may be a dict with keys as field @ids
        dataframes[record_set_id] = pd.DataFrame(records)

    # Show the first record set's columns
    example_record_set_id = record_set_ids[0]
    print(f"Loaded DataFrame for RecordSet @id: {example_record_set_id}\nColumns (@id):\n{dataframes[example_record_set_id].columns.tolist()}")
    dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For this example, we choose a numeric field (`MW`, molecular weight) by its `@id` (if present), filter for high values, and perform normalization.

In [ ]:
import numpy as np
# Pick a numeric field--try a few common ones
numeric_fields = [
    '@MW',    # molecular weight
    '@coverage',
    '@peptideCount',
    '@abundance',
    '@normalizedAbundance',
]

# Reuse previously loaded DataFrame
df = dataframes[example_record_set_id]

# Find a numeric field that exists
numeric_field_id = None
for f in numeric_fields:
    if f in df.columns:
        numeric_field_id = f
        break
if not numeric_field_id:
    # Otherwise, guess from dtypes
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if not numeric_field_id:
    print("No obvious numeric field found!")
else:
    print(f"Using numeric field {numeric_field_id}\n")
    # filter all non-null and >= threshold
    df_num = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df_num.mean() if not np.isnan(df_num.mean()) else 0
    filtered_df = df[df_num > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > mean:\n", filtered_df.head())
    filtered_df[f"{numeric_field_id}_normalized"] = (df_num - df_num.mean()) / df_num.std()
    print(f"\nNormalized {numeric_field_id} for filtered records:\n")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try a grouping field: find a string/categorical field
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:\n")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

> Here, we plot a histogram of the selected numeric field and (if available) a boxplot grouped by another attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    # Histogram
    plt.figure(figsize=(8,4))
    df_plot = pd.to_numeric(df[numeric_field_id], errors='coerce')
    df_plot = df_plot[np.isfinite(df_plot)]
    plt.hist(df_plot, bins=30, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group_field if available
    if group_field:
        plt.figure(figsize=(10,5))
        try:
            sns.boxplot(x=df[group_field], y=df[numeric_field_id])
            plt.xticks(rotation=45)
            plt.title(f"{numeric_field_id} by {group_field}")
            plt.show()
        except Exception as e:
            print(f"Could not plot boxplot: {e}")

## 6. Conclusion
We have demonstrated how to load, examine, and analyze the FAIR$^2$ dataset Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells using the `mlcroissant` library.

- We inspected the dataset structure with record sets and their fields (@id).
- We loaded data into DataFrames, selected numeric fields for analysis, performed basic filtering and normalization, and visualized results.

For more advanced use, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/) and explore the provided Croissant schema for additional metadata details.